In [1]:
import os

os.environ["ACCELERATE_TORCH_DEVICE"] = "cpu"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0.0" 

## Categorizando um produto

In [2]:
from transformers import pipeline

In [3]:
classificador = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [4]:
categorias = ['eletronics', 'food', 'toys', 'books']

In [5]:
predicao = classificador('''Latest model of smartphone with 5G connectivity and 128GB internal storage''', candidate_labels=categorias)

predicao

{'sequence': 'Latest model of smartphone with 5G connectivity and 128GB internal storage',
 'labels': ['eletronics', 'toys', 'food', 'books'],
 'scores': [0.814615786075592,
  0.06421393901109695,
  0.06153533235192299,
  0.0596349835395813]}

## Selecionando o modelo

In [ ]:
%pip install liqfit sentencepiece

In [8]:
from liqfit.pipeline import ZeroShotClassificationPipeline
from liqfit.models import T5ForZeroShotClassification
from transformers import T5Tokenizer

model = T5ForZeroShotClassification.from_pretrained('knowledgator/comprehend_it-multilingual-t5-base')
tokenizer = T5Tokenizer.from_pretrained('knowledgator/comprehend_it-multilingual-t5-base')
classifier = ZeroShotClassificationPipeline(model=model, tokenizer=tokenizer,
                                                      hypothesis_template = '{}', encoder_decoder = True, device = 'cpu')


You are using a model of type T5 to instantiate a model of type t5. This is not supported for all configurations of models and can yield errors.


In [9]:
descricao = '''Este water gel leve e refrescante, proporciona hidratação imediata que ajuda a aliviar o
repuxamento e aspereza da pele sensível'''

categorias_candidatas = ['beleza', 'cozinha', 'livros']

resultado = classifier(descricao, categorias_candidatas, multi_label=False)

resultado

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'sequence': 'Este water gel leve e refrescante, proporciona hidratação imediata que ajuda a aliviar o\nrepuxamento e aspereza da pele sensível',
 'labels': ['beleza', 'livros', 'cozinha'],
 'scores': [0.9420966506004333, 0.03873448818922043, 0.019168870523571968]}

In [10]:
import pandas as pd

In [11]:
resultado = pd.DataFrame(resultado).drop(['sequence'], axis=1)

resultado


,labels,scores
0,beleza,0.942097
1,livros,0.038734
2,cozinha,0.019169


In [12]:
descricao = '''
A fritadeira eletrica sem óleo start fry da elgin possui um design unico, capacidade para até 3,5 litros,
potência de 1400w e revestimento antiaderente. Seu sistema de circulação de ar ultrarapido frita e economiza
energia. Sua grelha de fritura é removivel e super fácil de limpar. Ela conta com uma proteção contra
super aquecimento. Possui controle de temperatura de 80°c a 200°c que permite você programar a temperatura
de preparo para cada tipo de alimento,timer para até 60 minutos com aviso sonoro e desligamento automático,
assim você pode deixar preparando sua refeição enquanto realiza outras tarefas.
'''

categorias_candidatas = ['beleza', 'cozinha', 'livros']

resultado = classifier(descricao, categorias_candidatas, multi_label=False)

resultado = pd.DataFrame(resultado).drop(['sequence'], axis=1)

resultado

,labels,scores
0,cozinha,0.718628
1,beleza,0.235084
2,livros,0.046288


## Aplicando o modelo aos dados

In [13]:
dados = pd.read_csv('dados/descricoes_produtos.csv')

dados[:5]

,Descrição
0,Liquidificador de alta potência com jarra de v...
1,"Forno Micro-ondas de 20 litros, com menu desco..."
2,Máquina de café espresso com reservatório de á...
3,Torradeira com capacidade para quatro fatias e...
4,"Panela elétrica multifuncional que cozinha, as..."


In [18]:
categorias = ['eletrodomesticos', 'eletronicos', 'beleza', 'brinquedos']

def categorizar(descricao):
    resultado = classifier(descricao, categorias, multi_label=False)
    categoria_max = max(zip(resultado['labels'], resultado['scores']), key=lambda x: x[1])[0]
    return categoria_max

dados['Categoria'] = dados['Descrição'].apply(categorizar)

In [21]:
dados[:5]

,Descrição,Categoria
0,Liquidificador de alta potência com jarra de v...,eletronicos
1,"Forno Micro-ondas de 20 litros, com menu desco...",eletrodomesticos
2,Máquina de café espresso com reservatório de á...,eletronicos
3,Torradeira com capacidade para quatro fatias e...,eletrodomesticos
4,"Panela elétrica multifuncional que cozinha, as...",eletrodomesticos
